In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported!')

In [ ]:
df = pd.read_csv('../data/tamil_nadu_waterborne_disease_data.csv')
print(f'✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

In [ ]:
def get_season(month):
    if month in [10, 11, 12]:
        return 'NE_Monsoon'
    elif month in [6, 7, 8, 9]:
        return 'SW_Monsoon'
    elif month in [3, 4, 5]:
        return 'Summer'
    else:
        return 'Winter'

df['season'] = df['month'].apply(get_season)
print('✅ Season column added!')
print(df['season'].value_counts())

In [ ]:
df = df.sort_values(['district', 'date']).reset_index(drop=True)

df['cases_last_month'] = df.groupby('district')[
    'total_disease_cases'].shift(1)
df['cases_2_months_ago'] = df.groupby('district')[
    'total_disease_cases'].shift(2)
df['rainfall_last_month'] = df.groupby('district')[
    'rainfall_mm'].shift(1)

df['cases_last_month'] = df['cases_last_month'].fillna(0)
df['cases_2_months_ago'] = df['cases_2_months_ago'].fillna(0)
df['rainfall_last_month'] = df['rainfall_last_month'].fillna(0)

print('✅ Lag features added!')
print(df[['district', 'date', 'total_disease_cases',
          'cases_last_month', 'cases_2_months_ago']].head(8))

In [ ]:
le_district = LabelEncoder()
df['district_encoded'] = le_district.fit_transform(df['district'])

le_season = LabelEncoder()
df['season_encoded'] = le_season.fit_transform(df['season'])

print('✅ Encoding done!')
print('\nDistrict Encoding:')
for i, district in enumerate(le_district.classes_):
    print(f'  {district} → {i}')
print('\nSeason Encoding:')
for i, season in enumerate(le_season.classes_):
    print(f'  {season} → {i}')

In [ ]:
feature_cols = [
    'rainfall_mm',
    'temperature_celsius',
    'pH',
    'turbidity_NTU',
    'coliform_count_per100ml',
    'dissolved_oxygen_mg_L',
    'total_dissolved_solids_mg_L',
    'month',
    'district_encoded',
    'season_encoded',
    'cases_last_month',
    'cases_2_months_ago',
    'rainfall_last_month'
]

target_col = 'outbreak_alert'

X = df[feature_cols]
y = df[target_col]

print(f'✅ Features selected: {len(feature_cols)}')
print(f'📊 X shape: {X.shape}')
print(f'🎯 y shape: {y.shape}')

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('✅ Features scaled successfully!')
print('\nSample scaled values:')
print(X_scaled.head(3).round(3))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('✅ Train-Test Split Done!')
print(f'\n📊 Training set: {X_train.shape[0]} rows')
print(f'📊 Testing set:  {X_test.shape[0]} rows')
print(f'\n🎯 Training labels:')
print(y_train.value_counts())
print(f'\n🎯 Testing labels:')
print(y_test.value_counts())

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

df.to_csv('../data/processed_data.csv', index=False)
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_district, '../models/le_district.pkl')
joblib.dump(le_season, '../models/le_season.pkl')

print('✅ All files saved!')
print('\n📁 Saved to data/:')
print('  - processed_data.csv')
print('  - X_train.csv, X_test.csv')
print('  - y_train.csv, y_test.csv')
print('\n🤖 Saved to models/:')
print('  - scaler.pkl')
print('  - le_district.pkl')
print('  - le_season.pkl')

In [ ]:
print('='*50)
print('📊 DAY 6 PREPROCESSING SUMMARY')
print('='*50)
print(f'✅ Total features used: {len(feature_cols)}')
print(f'✅ Training samples: {X_train.shape[0]}')
print(f'✅ Testing samples: {X_test.shape[0]}')
print(f'✅ Outbreak cases in train: {y_train.sum()}')
print(f'✅ Scaler + Encoders saved!')
print('\n🚀 Ready for ML Model Building tomorrow!')